# Tema 3.2 – Categorización de datos (World Happiness 2015–2024)

En este notebook se trabajará la **categorización de datos** usando el dataset unificado de *World Happiness* (2015–2024).

Nos enfocaremos en:

- Agrupar datos por **región** y por **año**.
- Crear categorías de **nivel de felicidad** (baja, media, alta) usando `pd.cut()`.
- Analizar cómo se distribuyen estas categorías entre regiones y a través del tiempo.

## Pregunta guía principal

> **¿Cómo se distribuyen los países del mundo en niveles de felicidad baja, media y alta, y qué regiones concentran más países en cada nivel?**

### Respuesta (visión general basada en los datos)

- Aproximadamente **21.8%** de las observaciones país-año se ubican en un nivel de felicidad **bajo**.
- Cerca de **61.0%** se ubican en un nivel de felicidad **medio**.
- Y alrededor de **17.2%** se ubican en un nivel de felicidad **alto**.

Cuando analizamos por región:
- **North America and ANZ** concentra el **100.0%** de sus observaciones en el nivel de felicidad **alto**.
- **Western Europe** también muestra un porcentaje elevado de países en el nivel alto (≈ **62.2%**).
- En contraste, **Sub-Saharan Africa** tiene alrededor de **58.2%** de sus observaciones en el nivel **bajo**, seguido de **South Asia** con ≈ **47.5%**.

Con esto, vemos que la categorización permite hacer lecturas rápidas sobre **brechas regionales** en felicidad.


## 0. Carga de librerías y unificación de los datos (2015–2024)

En esta sección repetimos, de forma resumida, el proceso de carga y unificación de los 10 archivos anuales, para que este notebook sea **autónomo**.


In [1]:
# =========================
# 0.1 Importar librerías
# =========================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)


In [2]:
# =========================
# 0.2 Cargar y unificar los datasets 2015–2024
# =========================

years = range(2015, 2025)  # 2015 a 2024
dfs = [] #esto va a ser una lista de df

for year in years:
    file_name = f"./data/world_happiness_{year}.csv"  # Ajusta el nombre si es necesario

    df_year = pd.read_csv(
        file_name,
        sep=";",              # Los archivos usan ';' como separador
        engine="python",      # Motor más flexible
        on_bad_lines="skip"    # Ignora filas defectuosas para evitar errores
    )

    df_year["Year"] = year
    dfs.append(df_year)

df = pd.concat(dfs, ignore_index=True)

df.head(10)

,Ranking,Country,Regional indicator,Happiness score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Year
0,1,Switzerland,Western Europe,7.59,8.26,0.96,73.0,0.99,0.37,0.24,2015
1,2,Iceland,Western Europe,7.56,7.70,1.00,73.0,0.94,0.55,0.74,2015
2,3,Denmark,Western Europe,7.53,7.84,0.97,70.0,0.97,0.43,0.12,2015
3,4,Norway,Western Europe,7.52,8.63,0.95,71.0,1.00,0.44,0.34,2015
4,5,Canada,North America and ANZ,7.43,7.85,0.94,71.0,0.95,0.58,0.40,2015
5,6,Finland,Western Europe,7.41,7.63,0.94,71.0,0.96,0.29,0.25,2015
6,7,Netherlands,Western Europe,7.38,7.86,0.91,71.0,0.92,0.60,0.42,2015
7,8,Sweden,Western Europe,7.36,7.88,0.92,72.0,0.99,0.46,0.21,2015
8,9,New Zealand,North America and ANZ,7.29,7.40,0.94,72.0,0.95,0.60,0.22,2015
9,10,Australia,North America and ANZ,7.28,7.89,0.93,72.0,0.97,0.55,0.35,2015


### 0.3 Limpieza inicial y creación de columna numérica de felicidad

Convertimos `Happiness score` a un valor numérico (`Happiness_score_num`) y renombramos algunas columnas para trabajar más cómodamente.


In [3]:
# =========================
# 0.3 Crear columna numérica de felicidad y renombrar columnas
# =========================

df["Happiness_score_num"] = (
    df["Happiness score"].astype(str).str.replace(",", ".", regex=False)
)
df["Happiness_score_num"] = pd.to_numeric(df["Happiness_score_num"], errors="coerce")

df = df.rename(columns={
    "Regional indicator": "Region",
    "GDP per capita": "GDP_per_capita",
    "Social support": "Social_support",
    "Healthy life expectancy": "Healthy_life_expectancy",
    "Freedom to make life choices": "Freedom",
    "Perceptions of corruption": "Corruption",
})

df.head()

,Ranking,Country,Region,Happiness score,GDP_per_capita,Social_support,Healthy_life_expectancy,Freedom,Generosity,Corruption,Year,Happiness_score_num
0,1,Switzerland,Western Europe,7.59,8.26,0.96,73.0,0.99,0.37,0.24,2015,7.59
1,2,Iceland,Western Europe,7.56,7.70,1.00,73.0,0.94,0.55,0.74,2015,7.56
2,3,Denmark,Western Europe,7.53,7.84,0.97,70.0,0.97,0.43,0.12,2015,7.53
3,4,Norway,Western Europe,7.52,8.63,0.95,71.0,1.00,0.44,0.34,2015,7.52
4,5,Canada,North America and ANZ,7.43,7.85,0.94,71.0,0.95,0.58,0.40,2015,7.43


## 1. Categorización por región

La primera forma de categorización será por **región** (`Region`). Vamos a agrupar los datos y calcular:

- Número de observaciones por región (`count`).
- Felicidad promedio (`mean`).
- Felicidad mediana (`median`).
- Dispersión (`std`).

### ¿Para qué agrupar por región?

- Para ver **diferencias estructurales** entre zonas geográficas del mundo.
- Para identificar qué regiones concentran países con niveles de felicidad más altos o más bajos.
- Para construir narrativas como: *“Occidente industrializado vs. regiones en desarrollo”*.

### Respuesta resumida

- Las regiones más felices en términos promedio son **North America and ANZ** y **Western Europe**.
- Las regiones con menores niveles promedio de felicidad son **Sub-Saharan Africa** y **South Asia**.
Esto ya sugiere una relación entre nivel de desarrollo y bienestar subjetivo medido.


In [4]:
# =========================
# 1.1 Resumen de felicidad por región
# =========================

region_summary = df.groupby("Region")["Happiness_score_num"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False)
region_summary

,count,mean,median,std
Region,,,,
North America and ANZ,40,7.151,7.192,0.172
Western Europe,232,6.723,6.870,0.744
Latin America and Caribbean,209,6.008,6.105,0.635
Central and Eastern Europe,141,5.743,5.766,0.582
East Asia,68,5.691,5.789,0.498
Southeast Asia,82,5.407,5.384,0.734
Commonwealth of Independent States,103,5.366,5.455,0.550
Middle East and North Africa,189,5.246,5.129,1.054
South Asia,65,4.398,4.500,0.853


## 2. Categorización en niveles de felicidad (baja, media, alta)

Ahora vamos a transformar la variable numérica `Happiness_score_num` en una **variable categórica** con tres niveles:

- `Low`  → felicidad baja  
- `Medium` → felicidad media  
- `High` → felicidad alta  

Usaremos `pd.cut()` para crear estos segmentos.

### ¿Por qué crear categorías?

- Facilita **interpretar** resultados para públicos no técnicos.
- Permite frases tipo: *“el 20% de los países está en felicidad baja”*.
- Ayuda a diseñar políticas: podemos enfocar atención en países (o regiones) de nivel bajo.

### Criterio utilizado

Sabemos que la felicidad mínima observada está cerca de **1.86** y la máxima en torno a **7.84**. Además, los cuartiles son aproximadamente:
- Q1 ≈ **4.59**
- Q2 (mediana) ≈ **5.43**
- Q3 ≈ **6.26**

Para simplificar en clase, definiremos los niveles así:

- **Low**: `Happiness_score_num < 4.5`
- **Medium**: `4.5 <= Happiness_score_num <= 6.5`
- **High**: `Happiness_score_num > 6.5`

Estos cortes están alineados con la distribución real del dataset y permiten una lectura intuitiva.


In [6]:
# =========================
# 2.1 Crear la variable categórica de nivel de felicidad
# =========================

bins = [0, 4.5, 6.5, 11]
labels = ["Low", "Medium", "High"]

df["Happiness_level"] = pd.cut(
    df["Happiness_score_num"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df[["Country", "Year", "Happiness_score_num", "Happiness_level"]].head(50)

,Country,Year,Happiness_score_num,Happiness_level
0,Switzerland,2015,7.59,High
1,Iceland,2015,7.56,High
2,Denmark,2015,7.53,High
3,Norway,2015,7.52,High
4,Canada,2015,7.43,High
5,Finland,2015,7.41,High
6,Netherlands,2015,7.38,High
7,Sweden,2015,7.36,High
8,New Zealand,2015,7.29,High
9,Australia,2015,7.28,High


### 2.2 Distribución global de niveles de felicidad

Una vez creada la variable categórica, podemos ver su distribución global.

### ✅ Respuesta global

- **21.8%** de las observaciones (país-año) están en felicidad **baja**.
- **61.0%** en felicidad **media**.
- **17.2%** en felicidad **alta**.

Es decir, la mayoría de observaciones se concentran en la zona media, con una proporción no despreciable de países en niveles bajos.


In [7]:
# =========================
# 2.2 Tablas de frecuencia globales
# =========================

level_counts = df["Happiness_level"].value_counts(dropna=False)
level_perc = df["Happiness_level"].value_counts(normalize=True, dropna=False) * 100

level_counts, level_perc

(Happiness_level
 Medium    908
 Low       329
 High      265
 Name: count, dtype: int64,
 Happiness_level
 Medium    60.453
 Low       21.904
 High      17.643
 Name: proportion, dtype: float64)

## 3. Niveles de felicidad por región

Ahora combinamos las dos ideas:

- **Agrupación por región**.
- **Categorización por nivel de felicidad**.

Así obtenemos, por ejemplo, el porcentaje de observaciones en nivel bajo/medio/alto **dentro de cada región**.

### Pregunta clave

> **¿Qué regiones concentran más países en nivel alto y cuáles tienen más países en nivel bajo?**

### Respuesta basada en los datos

- **North America and ANZ** tiene el porcentaje más alto de observaciones en nivel **alto** (≈ **100.0%**).
- **Western Europe** también destaca con ≈ **62.2%** en nivel alto.
- En el extremo opuesto, **Sub-Saharan Africa** tiene ≈ **58.2%** de observaciones en nivel **bajo**, seguida de **South Asia** con ≈ **47.5%**.

Esto refuerza la lectura de que **las regiones económicamente más desarrolladas concentran más países felices**, mientras que regiones con más desafíos estructurales acumulan más países en niveles bajos.


In [9]:
# =========================
# 3.1 Tablas de contingencia región vs nivel de felicidad
# =========================

#Tabla para el cruce
valid = df[df["Happiness_level"].notna()].copy()

region_level_counts = pd.crosstab(valid["Region"], valid["Happiness_level"])
region_level_perc = pd.crosstab(valid["Region"], valid["Happiness_level"], normalize="index") * 100

region_level_counts, region_level_perc

(Happiness_level                     Low  Medium  High
 Region                                               
 Central and Eastern Europe            2     126    13
 Commonwealth of Independent States   10      93     0
 East Asia                             0      63     5
 Latin America and Caribbean           6     173    30
 Middle East and North Africa         45     117    27
 North America and ANZ                 0       0    40
 South Asia                           32      33     0
 Southeast Asia                       15      63     4
 Sub-Saharan Africa                  216     154     0
 Western Europe                        3      83   146,
 Happiness_level                        Low  Medium     High
 Region                                                     
 Central and Eastern Europe           1.418  89.362    9.220
 Commonwealth of Independent States   9.709  90.291    0.000
 East Asia                            0.000  92.647    7.353
 Latin America and Caribbean      

## 4. Niveles de felicidad a lo largo del tiempo

También podemos ver cómo se distribuyen los niveles de felicidad por **año**, para responder:

> **¿Ha aumentado la proporción de países en nivel alto con el tiempo?**

A nivel descriptivo, esto se puede ver con una tabla de contingencia año vs nivel.


In [11]:
# =========================
# 4.1 Distribución de niveles por año
# =========================

year_level_perc = round(pd.crosstab(valid["Year"], valid["Happiness_level"], normalize="index") * 100, 2)
year_level_perc

Happiness_level,Low,Medium,High
Year,,,
2015,22.15,58.23,19.62
2016,26.75,54.78,18.47
2017,25.16,57.42,17.42
2018,26.45,59.35,14.19
2019,22.58,61.94,15.48
2020,18.42,66.45,15.13
2021,16.22,66.89,16.89
2022,17.24,64.83,17.93
2023,20.44,59.85,19.71


**Lectura sugerida (como docente):**

- Observa si la columna `High` gana peso en años recientes.
- Revisa si la columna `Low` pierde peso con el tiempo.
- Relaciona estos cambios con contextos globales (crisis, pandemias, crecimiento económico, etc.).


## 5. Tu turno – Práctica guiada 

Ahora es tu momento de aplicar estas ideas.


### Ejercicio 1 – Ajustar los umbrales de felicidad

1. Propón una nueva forma de definir los niveles de felicidad usando **cuartiles**:
   - Nivel 1: por debajo del Q1.
   - Nivel 2: entre Q1 y Q3.
   - Nivel 3: por encima del Q3.
2. Crea una nueva columna categórica (por ejemplo, `Happiness_level_q`).
3. Compara la distribución global de esta nueva categorización con la que usamos en el notebook.


In [ ]:
# ==== Ejercicio 1 – Tu solución aquí ====


### Ejercicio 2 – Enfoque en una región específica

1. Elige una región (por ejemplo, `Latin America and Caribbean`).
2. Filtra el DataFrame para esa región.
3. Calcula la distribución de niveles de felicidad (`Happiness_level`) en esa región.
4. Compara tus resultados con el promedio global y escribe un mini comentario.


In [ ]:
# ==== Ejercicio 2 – Tu solución aquí ====


### Ejercicio 3 – Niveles por año para una región

1. Usando la misma región del ejercicio anterior, crea una tabla año vs nivel de felicidad.
2. Identifica si hay años particularmente buenos o malos para esa región.
3. Escribe un breve análisis de 3–4 líneas.


In [ ]:
# ==== Ejercicio 3 – Tu solución aquí ====
